---
title: 'Computer Vision, Assignment 1: Elements of Projective Geometry'
jupyter:
  jupytext:
    text_representation:
      extension: .qmd
      format_name: quarto
      format_version: '1.0'
      jupytext_version: 1.19.5
  kernelspec:
    display_name: computer_vision_env
    language: python
    name: python3
---


In this assignment you will study the basics of projective geometry. You will study the representations of points lines and planes, as well as transformations and camera matrices.

Please see Canvas for detailed instructions on what is expected for a passing/higher grade. All computer exercises not marked **OPTIONAL** are mandatory to complete. 

### Submission Requirements:
Your final lab submission should include:
1. Your edited **notebook file** (`.ipynb`).
2. An **HTML printout** of the executed notebook with all outputs visible: File → Save and export Notebook As → HTML
3. A **pdf report** containing answers to the theoretical exercises (see separate document).


In [ ]:
# for creating responsive plots
# %matplotlib widget  

import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import cv2
from supplied import *

# Points in Homogeneous Coordinates

#### *Theoretical exercise 1* (see pdf)

## Computer Exercise 1
Write a function `pflat` that takes as input an array of shape $(m, n)$,
representing $n$ points in $\mathbb P^{m-1}$ in homogeneous coordinates. The output should be an $(m, n)$-array
containing the same homogeneous points, but where each point is normalized so that its last coordinate
is 1. (You may assume that none of the points have last homogeneous coordinate zero.) Apply the
function to the points in `x2D` and `x3D` in the file `compEx1.mat`, and plot the result. It will be useful
for later to also create a function `plot_points_2D` that takes as input a $(2, n)$-array representing n
two dimensional points in Cartesian coordinates and plots them. And a similar `plot_points_3D` for
three dimensional points.

**Useful python commands:**

`ax.view_init(40, 35)` # to view a 3D plot from a specific angle

`ax.set_aspect('equal')` # makes sure that all axes have the same scale

In [ ]:
def pflat(x:np.ndarray):
    """
    Normalizes array x of n homogeneous points to last coordinate 1
    
    Parameters
    ----------
    x : ndarray (m,n)
        Points in homogeneous coordinates.

    Returns
    -------
    y : ndarray (m,n)
        The same points in homogeneous coordinates, normalized so that 
        each point's last coordinate is 1.
    """

    # Your code here
    y = x/x[-1,:]

    return y


def plot_points_2D(points: np.ndarray, **kwargs):
    """
    Plots 2D points.
    
    Parameters
    ----------
    points : ndarray (2,n)
        Array representing n two dimensional points in Cartesian coordinates.
    """
    fig, ax = plt.subplots(1,1)
    ax.scatter(points[0,:], points[1,:], **kwargs)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect('equal')


def plot_points_3D(points: np.ndarray, ax=None, **kwargs):
    """
    Plot 3D points.

    Parameters
    ----------
    points : ndarray (3,n)
        Array containing n three-dimensional points in Cartesian coordinates.

    ax : matplotlib.axes.Axes, optional
        Matplotlib 3D axes on which to plot the points. If None, a new 3D axes 
        is created. Default is None.

    **kwargs
        Additional keyword arguments passed to ax.scatter3D() to customize the 
        appearance of the points, such as `c`, `marker` and `s`.
    """
    fig = plt.figure()
    if not ax:
        fig = plt.figure()
        ax = fig.add_subplot(projection="3d")

    assert ax, "axes cant be none"


    ax.scatter(*points, **kwargs)
    ax.set_aspect('equal')

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")


    # Your code here

In [ ]:
data = sp.io.loadmat('data/compEx1.mat')
x2D = data['x2D']
x3D = data['x3D']

x2D_flat = pflat(x2D)
plot_points_2D(x2D_flat[:-1,:])
plt.show()

x3D_flat = pflat(x3D)
plot_points_3D(x3D_flat[:-1,:])
plt.show()

# Lines

#### *Theoretical exercises: 2 and 3* (see pdf)

## Computer Exercise 2
Load and plot the image in `compEx2.jpg`. In the file `compEx2.mat` there are three pairs of image points.
Plot the image points in the same figure as the image. For each pair of points compute the line going through the points. 
Use the supplied function `rital` to plot the lines in the same image.
**Do these lines appear to be parallel (in 3D)?**
Compute the point of intersection between the second and third line (the lines obtained from the pairs `p2` and `p3`.
Plot this point in the same image.

In [ ]:
# Load data
# Your code here
data = sp.io.loadmat('./data/compEx2.mat')
p1 = data["p1"]
p2 = data["p2"]
p3 = data["p3"]
# Calculate equations for the lines l1, l2, l3
l1 = (sp.linalg.null_space(p1.T))
l2 = (sp.linalg.null_space(p2.T))
l3 = (sp.linalg.null_space(p3.T))
lines = np.hstack((l1, l2, l3)).T


# Calculate the intersection point (pi) between lines l2 and l3
pi = pflat(sp.linalg.null_space(lines[1:3]))

# Show the image and plot the points in the image 
# Your code here
plt.figure()
plt.scatter(p1[0],p1[1], label="p1")
plt.scatter(p2[0],p2[1], label = "p2")
plt.scatter(p3[0],p3[1], label="p3")

# Plot the lines in the image
# Plot the intersection point in the image

plt.scatter(pi[0],pi[1], color = "blue", label= "cross between l2 & l3")
rital(lines)
# Your code here

plt.legend()

The distance between a 2D-point $x=(x_1,x_2)$ in Cartesian coordinates and a line $l = (a,b,c)$ can be computed using the distance formula

\begin{equation}
    d = \frac{|ax_1+bx_2+c|}{\sqrt{a^2+b^2}},
\end{equation}

see your linear algebra book.
Implement a function `point_line_distance_2D` that computes the distance between a point and a line.
Compute the distance between the first line and the intersection point. **Is it close to zero? Why/why not?**

In [ ]:
def point_line_distance_2D(point, line):
    """
    Implement the distance formula.
    
    Parameters
    ----------
    point : ndarray (3,)
    line : ndarray (3,)

    Returns
    -------
    distance : float
        The distance between the point and line.

    """

    # Your code here
    distance = np.linalg.norm(line*point)/np.sqrt(np.sum(line**2))

    return distance

In [ ]:
dist = point_line_distance_2D(pi, l1) # Computes the distance between the intersection point (pi) and the first line (l1).
print(f"Distance: {dist}")

### Your answer here: 

(*Include answers to all questions marked in bold.*)
#
The lines does not seem to be parallel, in 3 dimentions. Because the points all lay in the image plane they at least share one dimention but, the slope of them does not seem to be parallel.

The gap seem pretty small but this is just by chance. This is because this line is independent och the points p2 and p3, because it was only created by the points in p1 and the point $\pi$ was created by the intersection of the l2 and l3 lines which depends on the points p2 and p3. If we choose different points for p1 we could do the distance arbitrary large.


# Projective Transformations

#### *Theoretical exercises: 4, 5, 6* (see pdf)

# The Pinhole Camera

#### *Theoretical exercise 7* (see pdf)

## Computer Exercise 3
Load and plot the images `compEx3im1.jpg` and `compEx3im2.jpg`.
The file `compEx3.mat` contains the camera matrices `P1`, `P2` and a point model `U` of the statue.

In order to visualize cameras in 3D it is useful to plot them as vectors pointing from the camera center in the direction of the principal axis.
Implement a function `camera_center_and_axis` that takes a camera matrix as input and outputs
the camera center and principal axis.
The camera center should be in Cartesian coordinates and the principal axis should be normalized to length 1.
Compute the camera centers and principal axes of the two cameras.

In [ ]:
def camera_center_and_axis(P):
    """
    Computes the camera center and principal axis from a camera matrix P.
    
    Parameters
    ----------
    P : ndarray (3,4)
        Camera matrix.

    Returns
    -------
    camera_center : ndarray (3,)
        Camera center in Cartesian coordinates.

    principal_axis : ndarray (3,)
        Principal axis normalized to length 1.
    """
    
    # Your code here

    camera_center=pflat(sp.linalg.null_space(P))[:-1]

    principal_axis_direction = (P[2,:3]*np.linalg.det(P[:,:3])).reshape(3,1)
    principal_axis = principal_axis_direction/np.linalg.norm(principal_axis_direction)
    assert camera_center.shape == principal_axis.shape

    return camera_center, principal_axis

In [ ]:
data = sp.io.loadmat('data/compEx3.mat')
U = data['U']
P1 = data['P1']
P2 = data['P2']
img1 = plt.imread('data/compEx3im1.jpg')
img2 = plt.imread('data/compEx3im2.jpg')

plt.figure(4) 
plt.imshow(img1, cmap='gray')
plt.figure(5)
plt.imshow(img2, cmap='gray')

# Compute the camera centers and principal axes of the two cameras
C1, pr1 = camera_center_and_axis(P1)
print(f"Camera center 1:\n{C1}")
print(f"\nPrincipal axis 1:\n{pr1}")

C2, pr2 = camera_center_and_axis(P2)
print(f"\nCamera center 2:\n{C2}")
print(f"\nPrincipal axis 2:\n{pr2}")

Next, implement a function `plot_camera` which takes a camera matrix and a scale $s$ as input and plots the principal axis of the camera scaled by $s$, from the camera center. 
Plot the 3D-points in `U` and the camera centers in the same 3D plot
(make sure that the 4th coordinate of `U` is one before you plot by using `pflat`.

**Useful python commands:**

`mpl_toolkits.mplot3d.axes3d.Axes3D.quiver` # can be used to plot arrows in a 3D plot

In [ ]:
def plot_camera(camera_matrix, s,ax=None,**kwargs ):
    """
    Plots the principal axis from the camera center, scaled by s.
    
    Parameters
    ----------
    camera_matrix : ndarray (3,4)

    s : float
        Scale factor for the principal axis vector, determining
        the length of the plotted arrow.

    camera_name : str
        Name of the camera position
    
    ax : mpl_toolkits.mplot3d.axes3d.Axes3D, optional
        Matplotlib 3D axes on which to plot the camera. If None,
        a new 3D axes is created. Default is None.
    """

    # Your code here
    camera_center, principal_axis=camera_center_and_axis(camera_matrix)
    if ax:
        ax.scatter(*camera_center, color="black")
        ax.quiver(*camera_center,*principal_axis, length=s , **kwargs)
    else:
        fig = plt.figure()
        ax =fig.add_subplot(projection="3d")
        plt.arrow(*camera_center, color="black")
        plt.quiver(*camera_center,*principal_axis, length=s ,**kwargs )


In [ ]:
# Plot the 3D-points in U. Also plot the camera centers and principal axes in the same 3D plot.

# Your code here
#print(U)
fig = plt.figure(figsize = (8,8))
ax =fig.add_subplot(projection="3d")
ax.set_aspect('equal')
points_reshaped = pflat(U)[:-1]
plot_points_3D(points_reshaped, ax, color="blue", s=1, alpha=0.4)
plot_camera(P1,10, ax, label="Camera 1", color= "orange")
plot_camera(P2,10, ax, label="Camera 2", color="red")
ax.view_init(10,-150, vertical_axis ='x')
fig.legend()
fig.show()

Finally, project the points in `U` into the cameras `P1` and `P2` and plot the result in the same plots as the images.
**Does the result look reasonable?** 

In [ ]:
# Your code here

plot_points_2D(pflat(P1@U)[:-1],s = 0.5, alpha = 0.3)
plt.imshow(img1, cmap='gray')
plt.show()

plot_points_2D(pflat(P2@U)[:-1], s= 0.5, alpha=0.3)
plt.imshow(img2, cmap='gray')

### Your answer here: 

(*Include answers to all questions marked in bold.*)
#


#### *Theoretical exercise 8* (OPTIONAL, 10 optional points, see pdf)

## Computer Exercise 4 (OPTIONAL, 15 optional points)
Load and plot the image compEx4.jpg, which is an image of a poster. (Here the axes units are pixels.)
The file `compEx4.mat` contains the inner parameters `K`, the corner points of the poster and the 3D plane `v` that contains the poster.
The camera matrix for the camera that generated this image is 
\begin{equation}
P_1 = K [ I \ 0].
\end{equation}
The goal of this exercise is to create an image of the poster taken by a camera 
2.5m to the right of the original camera.

Start by plotting the corner points and the image in the same 2D-figure. (Make sure to set equal aspect ratio.) Note the scale on the axis. **Where is the origin of the image coordinate system located?**

In [ ]:
# Your code here

#### Your answer here: 
#

To be able to use the formulas derived in Theoretical Exercise 8, we must first ensure that we have calibrated cameras.
To do this, normalize the corner points by multiplying with $K^{-1}$ and plot them in a new 2D-figure. Use `ax.yaxis.set_inverted(True)` to make the y-axis point downwards (as for the previous image) and set equal aspect ratio.
Note the difference in scale compared to the previous figure. **Where is the origin of the image coordinate system located?**

In [ ]:
# Your code here

#### Your answer here: 
#

Since the inner parameters $K$ have been removed our calibrated camera is $[I \ 0]$.
Using the results from Theoretical Exercise 8, compute the 3D points in the plane `v` that project onto the corner points.
Compute the camera center and principal axis, and plot together with the 3D-points. Use the supplied function `axis3D_equal` to ensure equal aspect ratio. Does it look reasonable?

In [ ]:
# Compute the 3D points in the plane v that project onto the corner points
# Your code here

# Compute the camera center and principal axis for camera 1, and plot the camera together with the 3D points
# Your code here

Next, compute a new camera with camera center in $(2.5,0,0)$ (2.5 units to the right of $P_1$)
and orientation 
\begin{equation}
R = \left( \begin{array}{ccc}
\cos(\pi/5) & 0 & \sin(\pi/5) \\
0 & 1 & 0 \\
-\sin(\pi/5) & 0 & \cos(\pi/5)
\end{array} \right)
\end{equation}
($-36$ degrees rotation around the y-axis. Note that the y-axis points down in the first image.)
Compute the new camera and plot in the same figure. (Don't forget to ensure equal aspect ratio.)

In [ ]:
# Compute P2, the camera matrix for camera 2

# Your code here
P2 = None

# Print the camera matrix
print(f"Camera matrix 2:\n{P2}")

# Plot the 3D points and both cameras in the same plot
# Example code:
# plt.figure(12)
# ax = plt.axes(projection='3d')
# ax.view_init(-30, -100)
# plot_points_3D(corners_3D, ax, s=20)
# plot_camera(P1, 1, ax)
# plot_camera(P2, 1, ax)
# axis3D_equal()
# plt.show()

# Your code here 

Compute the homography in Theoretical Exercise 8 and transform the normalized corner points
to the new (virtual) image. Plot the transformed points (don't forget to divide by the third coordinate) in a new 2D-figure.
**Does the result look like you would expect it to when moving the camera like this?**
Also, project the 3D points into the same image using the camera matrix. Does this give the same result?

In [ ]:
# Compute the homography in TE8 and transform the normalized corner points to the new (virtual) image
# Your code here

# Plot the transformed corners in a new figure
# Your code here

# Also project the 3D points into the same image using the camera matrix
# Your code here

#### Your answer here: 
#

We now have a homography that takes normalized points in the first camera and transforms them to normalized points in the second camera.
To remove the need for normalization we simply include the normalization in the homography.
If $\tilde{x} = K^{-1} x$ and $\tilde{y} = K^{-1} y$ then for the normalized points $\tilde{x}$ and $\tilde{y}$ we have
\begin{equation}
\tilde{y} \sim H \tilde{x} \Leftrightarrow  K^{-1}y \sim H K^{-1} x \Leftrightarrow y \sim K H K^{-1} x
\end{equation}
Therefore the total transformation is $H_{tot} = KHK^{-1}$.
Transform the original image and the corner points using the homography $H_{tot}$ and plot both in a new 2D-figure.

**Useful python commands:**

`transformed_img = cv2.warpPerspective(img, H_tot, (img.shape[1], img.shape[0]))` # can be used to apply the homography H_tot to the image

In [ ]:
# Transform the original image and the corner points using the homography H_tot
# Your code here

# Plot transformed image and corner points
# Your code here